# Initialize Spark and importing a dataset from the virtual campus

In [0]:
from pyspark.sql import SparkSession


spark = SparkSession.builder.appName("Step3 wordcounter").getOrCreate()

#loading the data
wrd = spark.read.table("workspace.default.annex_1_aplication")


print(f"Total rows: {wrd.count()}")
display(wrd.limit(10))

#Performing ETL tasks (Extract,Transform, Load) using PySpark 

In [0]:
from pyspark.sql.functions import col, initcap, explode, split, trim, substring, when

# Remove rows with null values
wrd_clean = wrd.filter(col("value").isNotNull())

# Convert the first letter of each word to uppercase
wrd_clean = wrd_clean.withColumn("value", initcap(col("value")))

# Identify if the first letter of each word is a vowel or consonant and remove empty strings
wrd_clean = wrd_clean.withColumn(
    "first_letter_type",
    when(
        substring(col("value"), 1, 1).isin("A", "E", "I", "O", "U"),
        "vowel"
    ).otherwise("consonant")
)
wrd_clean = wrd_clean.filter(trim(col("value")) != "")

# Display the result
display(wrd_clean.limit(10))
print(f"Total rows: {wrd_clean.count()}")

In [0]:
from pyspark.sql.functions import substring, col

# Extract the first letter from each word
wrd_first_letter = wrd_clean.withColumn("first_letter", substring(col("value"), 1, 1))

# Display the result
display(wrd_first_letter.limit(20))

#Running  SQL  queries  using  Spark  SQL to  analyze  textual  or numeric data 

In [0]:
wrd_clean.createOrReplaceTempView("wrd_clean_view")

# Count words starting with a vowel
result = spark.sql("""
    SELECT first_letter_type, COUNT(*) AS count
    FROM wrd_clean_view
    GROUP BY first_letter_type
""")

display(result)

In [0]:
from pyspark.sql.functions import substring, col

# Count occurrences of each initial letter
initial_letter_count = wrd_clean.withColumn("first_letter", substring(col("value"), 1, 1)) \
    .groupBy("first_letter") \
    .count() \
    .orderBy("count", ascending=False)

display(initial_letter_count)

In [0]:
from pyspark.sql.functions import length

# Count occurrences of each word length
word_length_count = wrd_clean.withColumn("word_length", length(col("value"))) \
    .groupBy("word_length") \
    .count() \
    .orderBy("word_length")

display(word_length_count)

In [0]:
from pyspark.sql.functions import col

# Count occurrences of each word
word_count = wrd_clean.groupBy(col("value")).count().orderBy("count", ascending=False)

display(word_count)

#Generating a visualization or summary output 

In [0]:
import matplotlib.pyplot as plt

# Query 1: Count words starting with a vowel or consonant
result_pd = result.toPandas()
plt.figure(figsize=(6,4))
plt.bar(result_pd['first_letter_type'], result_pd['count'], color=['skyblue', 'salmon'])
plt.title('Word Count by First Letter Type')
plt.xlabel('First Letter Type')
plt.ylabel('Count')
plt.show()

# Query 2: Count occurrences of each initial letter (Pie Chart)
initial_letter_count_pd = initial_letter_count.toPandas()
plt.figure(figsize=(8,8))
plt.pie(initial_letter_count_pd['count'], labels=initial_letter_count_pd['first_letter'], autopct='%1.1f%%', colors=plt.cm.Paired.colors)
plt.title('Word Count by Initial Letter')
plt.show()

# Query 3: Count occurrences of each word length
word_length_count_pd = word_length_count.toPandas()
plt.figure(figsize=(10,5))
colors = plt.cm.tab20.colors[:len(word_length_count_pd)]
plt.bar(word_length_count_pd['word_length'], word_length_count_pd['count'], color=colors)
plt.title('Word Count by Word Length')
plt.xlabel('Word Length')
plt.ylabel('Count')
plt.show()

# Query 4: Count occurrences of each word (top 20), ordered by count descending
word_count_pd = word_count.toPandas().sort_values('count', ascending=False).head(20)
plt.figure(figsize=(12,6))
plt.bar(word_count_pd['value'], word_count_pd['count'], color='purple')
plt.title('Top 20 Most Frequent Words')
plt.xlabel('Word')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

# Summary Output
The  notebook performed a data extraction, transformation, and classification process using PySpark. Initially, a dataset containing 300 rows was loaded from the workspace.default.annex_1_application table. During the ETL process, rows with null values and empty strings were removed, and the first letter of each word in the “value” column was capitalized. A new column called first_letter_type was then added to classify each word based on whether it begins with a vowel or a consonant. The final results included examples such as “Speech” and “Vision” classified as consonants, and “Assistants” classified as a vowel. Although an initial ImportError occurred while importing SparkSession, the remaining commands executed successfully, allowing the data to be processed and the final results to be displayed.